# Select proteins by EC number

Choose an EC hierarchy level and a maximum number of proteins per EC number. This notebook lists UniProt proteins with PDB cross-references, excludes accessions assigned to multiple EC classes at the selected level, maps each chosen accession to a PDB chain, and saves a chain-level manifest.

The default source filter is Homo sapiens, matching the recovered experiment notes. Set HUMAN_ONLY to False to include other organisms. PDB chain mapping is retrieved from PDBe; structure files are downloaded from RCSB in the next notebook.

Data sources: [UniProt REST API](https://www.uniprot.org/api-documentation), [PDBe API](https://www.ebi.ac.uk/pdbe/pdbe-rest-api), and the [RCSB PDB file service](https://www.rcsb.org/docs/programmatic-access/file-download-services). The EC list is derived from PDB-linked UniProt accessions, so it reflects classes with structure cross-references rather than every entry in the Enzyme Nomenclature database.

In [1]:
from pathlib import Path
import json
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from enzyme_classification.rcsb import (
    ec_protein_counts,
    fetch_uniprot_ec_pdb_records,
    select_ec_protein_manifest,
)

DATA_DIR = PROJECT_ROOT / "data"
INTERIM_DIR = DATA_DIR / "interim"
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

# User controls: EC level 1-4 and maximum UniProt proteins to select per EC.
EC_LEVEL = 1
PROTEINS_PER_EC = 5
HUMAN_ONLY = True
SELECTED_EC_NUMBERS = None  # e.g. ["1", "2"] at level 1, or None for every available class.
RANDOM_STATE = 42
REFRESH_UNIPROT_CACHE = False

UNIPROT_CACHE = INTERIM_DIR / f"uniprot_ec_pdb_human_{HUMAN_ONLY}.tsv"
MANIFEST_PATH = INTERIM_DIR / f"ec_protein_manifest_level_{EC_LEVEL}.tsv"
MANIFEST_METADATA_PATH = INTERIM_DIR / f"ec_protein_manifest_level_{EC_LEVEL}.json"
PDBE_MAPPING_CACHE = INTERIM_DIR / "pdbe_uniprot_chain_mappings.json"
REBUILD_MANIFEST = False
MANIFEST_SETTINGS = {
    "ec_level": EC_LEVEL,
    "proteins_per_ec": PROTEINS_PER_EC,
    "human_only": HUMAN_ONLY,
    "selected_ec_numbers": SELECTED_EC_NUMBERS,
    "random_state": RANDOM_STATE,
}

records = fetch_uniprot_ec_pdb_records(
    UNIPROT_CACHE,
    human_only=HUMAN_ONLY,
    refresh=REFRESH_UNIPROT_CACHE,
)
print(f"Loaded {len(records):,} UniProt records from {UNIPROT_CACHE}")

Loaded 2,811 UniProt records from ./data/interim/uniprot_ec_pdb_human_True.tsv


In [2]:
available_ec_numbers = pd.DataFrame(ec_protein_counts(records, EC_LEVEL))
if available_ec_numbers.empty:
    raise RuntimeError(f"No EC classes with PDB-linked proteins found at level {EC_LEVEL}")

display(available_ec_numbers)
print(f"Available EC numbers at level {EC_LEVEL}: {len(available_ec_numbers)}")

,ec_number,uniprot_proteins_with_pdb
0,1,321
1,2,1068
2,3,1004
3,4,76
4,5,95
5,6,57
6,7,66


Available EC numbers at level 1: 7


cached_settings = None
if MANIFEST_METADATA_PATH.exists():
    cached_settings = json.loads(MANIFEST_METADATA_PATH.read_text(encoding="utf-8")).get("settings")

if (MANIFEST_PATH.exists() and cached_settings == MANIFEST_SETTINGS
        and not REBUILD_MANIFEST and not REFRESH_UNIPROT_CACHE):
    manifest = pd.read_csv(MANIFEST_PATH, sep="\t", dtype=str).fillna("")
    print(f"Reusing saved manifest: {MANIFEST_PATH}")
else:
    manifest_rows = select_ec_protein_manifest(
        records,
        level=EC_LEVEL,
        proteins_per_ec=PROTEINS_PER_EC,
        selected_ec_numbers=SELECTED_EC_NUMBERS,
        random_state=RANDOM_STATE,
        mapping_cache_path=PDBE_MAPPING_CACHE,
        output_path=MANIFEST_PATH,
        verbose=True,
    )
    manifest = pd.DataFrame(manifest_rows)
    metadata = {
        "settings": MANIFEST_SETTINGS,
        "sample_count": len(manifest),
    }
    temporary_metadata_path = MANIFEST_METADATA_PATH.with_suffix(".json.tmp")
    temporary_metadata_path.write_text(json.dumps(metadata, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    temporary_metadata_path.replace(MANIFEST_METADATA_PATH)
    print(f"Saved manifest: {MANIFEST_PATH}")

print(f"Selected {len(manifest):,} chain-level samples")
display(manifest.groupby("ec_number").size().rename("selected_proteins").to_frame())
display(manifest.head())

In [3]:
manifest_rows = select_ec_protein_manifest(
    records,
    level=EC_LEVEL,
    proteins_per_ec=PROTEINS_PER_EC,
    selected_ec_numbers=SELECTED_EC_NUMBERS,
    random_state=RANDOM_STATE,
    mapping_cache_path=PDBE_MAPPING_CACHE,
    output_path=MANIFEST_PATH,
    verbose=True,
)
manifest = pd.DataFrame(manifest_rows)
print(f"Saved {len(manifest):,} chain-level samples to {MANIFEST_PATH}")
display(manifest.groupby("ec_number").size().rename("selected_proteins").to_frame())
display(manifest.head())

EC 1: selected 5/5 proteins
EC 2: selected 5/5 proteins
EC 3: selected 5/5 proteins
EC 4: selected 5/5 proteins
EC 5: selected 5/5 proteins
EC 6: selected 5/5 proteins
EC 7: selected 5/5 proteins
Saved 35 chain-level samples to ./data/interim/ec_protein_manifest_level_1.tsv


,selected_proteins
ec_number,
1,5
2,5
3,5
4,5
5,5
6,5
7,5


,sample_id,pdb_id,entity_id,label_asym_id,auth_asym_id,mapping_identity,mapping_coverage,ec_number,ec_level,uniprot_accession,ec_numbers,protein_name,organism
0,4b4o.H,4b4o,1,H,H,0.99,0.993,1,1,Q9NRG7,1.1.1.-,Epimerase family protein SDR39U1 (EC 1.1.1.-) ...,Homo sapiens (Human)
1,3lc4.B,3lc4,1,B,B,1.00,0.937,1,1,P05181,1.14.13.n7;1.14.14.1,Cytochrome P450 2E1 (EC 1.14.14.1) (4-nitrophe...,Homo sapiens (Human)
2,6sy1.B,6sy1,1,B,B,1.00,0.952,1,1,Q96HY7,1.2.4.-,2-oxoadipate dehydrogenase complex component E...,Homo sapiens (Human)
3,7mjc.B,7mjc,1,B,B,1.00,0.963,1,1,P30837,1.2.1.28;1.2.1.3;1.2.1.36;1.7.1.-,"Aldehyde dehydrogenase family 1 member B1, mit...",Homo sapiens (Human)
4,2dfd.D,2dfd,1,D,D,1.00,0.947,1,1,P40926,1.1.1.-;1.1.1.37,"Malate dehydrogenase, mitochondrial (EC 1.1.1....",Homo sapiens (Human)
